# SeismicX-Cont Mini Quickstart

This notebook demonstrates the small two-hour SeismicX-Cont subset: file checks, annotation inspection, SQLite waveform queries, HDF5 reading, dataloader usage, and picker/evaluation command templates.

Run it from the `publish_mini/` folder or from `publish_mini/notebooks/`.

## 1. Mini Subset

- Dense hour: `2019-07-06T04:00:00Z` to `2019-07-06T05:00:00Z`, 6948 reference picks.
- Quiet nonzero hour: `2021-11-14T16:00:00Z` to `2021-11-14T17:00:00Z`, 7 reference picks.

The mini subset is for quick trials and software smoke tests. Use the full SeismicX-Cont release for benchmark-scale evaluation.

In [ ]:
from pathlib import Path
import json
import sqlite3
import sys

import h5py
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

H5_DIR = ROOT / "data" / "hdf5"
INDEX_DB = ROOT / "data" / "index" / "waveform_index.sqlite"
ANNOTATION_JSON = ROOT / "data" / "label" / "annotations_mini_two_hours.json"
PICKERS_DIR = ROOT / "pickers"
PICKS_DIR = ROOT / "data" / "picks"

paths = {
    "HDF5 directory": H5_DIR,
    "SQLite waveform index": INDEX_DB,
    "Mini annotation JSON": ANNOTATION_JSON,
    "Picker models": PICKERS_DIR,
}

for name, path in paths.items():
    status = "OK" if path.exists() else "MISSING"
    size = f"{path.stat().st_size / 1024**2:.1f} MiB" if path.is_file() else ""
    print(f"{name:24s} {status:8s} {path} {size}")

print("\nHDF5 files:")
for path in sorted(H5_DIR.glob("*.h5")):
    print(f"  {path.name:44s} {path.stat().st_size / 1024**2:8.1f} MiB")


## 2. Inspect The Annotation Subset

In [ ]:
with ANNOTATION_JSON.open("r", encoding="utf-8") as f:
    annotations = json.load(f)

print(json.dumps(annotations["summary"], ensure_ascii=False, indent=2))

print("\nWindows:")
for item in annotations["subset_windows"]:
    print(f"  {item['name']}: {item['starttime']} -> {item['endtime']}")


## 3. Query The SQLite Waveform Index

In [ ]:
conn = sqlite3.connect(INDEX_DB)
conn.row_factory = sqlite3.Row

for table in ["hdf5_files", "stations", "waveform_segments"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:18s}: {n:,}")

rows = conn.execute(
    """
    SELECT h5_file, dataset_path, network, station, location, channel,
           starttime, endtime, sampling_rate, npts
    FROM waveform_segments
    WHERE starttime <= ? AND endtime >= ?
    ORDER BY network, station, location, channel
    LIMIT 5
    """,
    ("2019-07-06T04:00:10", "2019-07-06T04:00:00"),
).fetchall()

for row in rows:
    item = dict(row)
    print(f"{item['network']}.{item['station']}.{item['location']} {item['channel']} {item['starttime']} -> {item['endtime']} path={item['dataset_path']}")

conn.close()


## 4. Read And Plot One HDF5 Segment

In [ ]:
h5_path = sorted(H5_DIR.glob("continuous_waveform_usa_20190706_04.h5"))[0]

with h5py.File(h5_path, "r") as h5:
    # Pick the first vertical component when available.
    candidates = []
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset) and name.split("/")[-2].endswith("Z"):
            candidates.append(name)
    h5.visititems(visitor)
    dataset_path = candidates[0]
    ds = h5[dataset_path]
    sr = float(ds.attrs["sampling_rate"])
    data = ds[: int(min(len(ds), 60 * sr))]
    starttime = ds.attrs["starttime"]
    channel = ds.attrs["channel"]
    station = f"{ds.attrs['network']}.{ds.attrs['station']}.{ds.attrs['location']}"

t = np.arange(len(data)) / sr
plt.figure(figsize=(10, 3))
plt.plot(t, data, lw=0.7)
plt.title(f"{station} {channel} from {starttime}")
plt.xlabel("Seconds")
plt.ylabel("Counts")
plt.tight_layout()


## 5. Use The PyTorch Dataloader

The dataloader groups station-hour waveform channels into picker-ready samples. It uses the same loader as the full benchmark.

In [ ]:
try:
    from torch.utils.data import DataLoader
    from utils.hdf5_waveform_dataset import HDF5WaveformDataset, waveform_collate_fn

    dataset = HDF5WaveformDataset(
        h5_file=str(H5_DIR / "continuous_waveform_usa_*.h5"),
        mode="three",
        allowed_families=("HH", "BH", "EH", "HN"),
        allowed_z_only_channels=("EHZ",),
        allow_z_only=True,
        replicate_z_only=True,
        target_sampling_rate=100.0,
    )
    print("HDF5 files:", len(dataset.h5_files))
    print("Dataloader samples:", len(dataset))

    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=waveform_collate_fn)
    item = next(iter(loader))[0]
    print("station_id:", item["station_id"])
    print("channels:", item["channels"])
    print("starttime:", item["starttime"])
    print("waveform shape:", tuple(item["waveform"].shape))
    print("sampling_rate:", item["sampling_rate"])
    dataset.close()
except Exception as exc:
    print("Dataloader example skipped or failed:", repr(exc))


## 6. Run A Picker

The mini package includes small example picker models in `pickers/`. Running a picker can take a few minutes because it scans station-hour samples from both HDF5 files. The command below writes JSONL picks that can be evaluated by the next section.

In [ ]:
picker_cmd = [
    "python", "scripts/run_picker_to_jsonl.py",
    "--h5_input", "data/hdf5/continuous_waveform_usa_*.h5",
    "--picker_model", "pickers/phasenet.jit",
    "--output_jsonl", "data/picks/phasenet.mini.phase.jsonl",
    "--device", "cpu",
    "--canonical_input_length", "360000",
    "--max_picks_per_sample", "0",
    "--no_auto_restart",
]

print("Run from publish_mini/:\n")
print(" \\\n  ".join(picker_cmd))


## 7. Evaluate Picker Output

After generating a JSONL pick file, use the mini annotation JSON and mini waveform index. The pick-index SQLite file should be written to local temporary storage first on systems where external drives are unreliable for SQLite writes.

In [ ]:
pick_jsonl = PICKS_DIR / "phasenet.mini.phase.jsonl"
eval_cmd = [
    "python", "scripts/evaluate_picks.py",
    "--auto-jsonl", str(pick_jsonl),
    "--label-json", str(ANNOTATION_JSON),
    "--index-db", "/tmp/seismicx_cont_mini_phasenet.sqlite",
    "--outdir", "eval_picks/phasenet_mini",
    "--waveform-db", str(INDEX_DB),
    "--tp-tol", "1.5",
    "--err-window", "5.0",
    "--build-index",
    "--drop-existing",
    "--plot",
]

print("Evaluation command:\n")
print(" \\\n  ".join(eval_cmd))

if pick_jsonl.exists():
    print(f"\nPick file exists: {pick_jsonl}")
else:
    print("\nPick file does not exist yet; run the picker command first.")


## 8. Optional Consensus Candidates

If several picker JSONL files are available, build a consensus-supported candidate layer. This is an exploratory audit product, not the primary benchmark label definition.

In [ ]:
consensus_cmd = [
    "python", "scripts/build_consensus_picks_json.py",
    "--auto-jsonl", "data/picks/phasenet.mini.phase.jsonl", "data/picks/pnsn_v3_diff.mini.phase.jsonl", "data/picks/skynet.mini.phase.jsonl",
    "--label-json", str(ANNOTATION_JSON),
    "--index-db", "/tmp/seismicx_cont_mini_consensus.sqlite",
    "--out-json", "data/label/consensus_mini_picks.json",
    "--build-index",
    "--drop-existing",
    "--is-phase", "1.5",
    "--min-models", "3",
    "--human-match", "1.5",
]

print("Consensus command:\n")
print(" \\\n  ".join(consensus_cmd))
